# Build the search index on a cloud GPU (Colab / Kaggle)

Index building is the only heavy step of this project: it decodes the Act's 1,098-page PDF and
embeds ~2,900 statutory passages with `BAAI/bge-base-en-v1.5`. On a CPU-only laptop that takes
~12 minutes at full load; on a free Colab T4 GPU it takes about a minute.

This notebook runs **exactly the same code** as `backend/scripts/ingest.py`. Its output (`index_bundle.zip`)
is imported on the laptop with `python scripts/import_index.py index_bundle.zip`, after which the laptop only
does the light per-question work.

**Before you start**
1. On your laptop, from `backend/`: `python scripts/make_colab_bundle.py` → creates `backend/colab_bundle.zip`
   (code + the official PDF, ~7 MB; your `.env` and API key are never included).
2. In Colab: **Runtime → Change runtime type → T4 GPU**. On Kaggle: Settings → Accelerator → GPU, and add
   `colab_bundle.zip` as a dataset.

No API key is needed here: indexing uses only the local embedding model.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - this will still work, just slower (CPU)."

## 2. Get the code and the PDF
Upload `colab_bundle.zip` when prompted (Colab), or attach it as a dataset (Kaggle).

In [ ]:
import glob, os, shutil, zipfile
WORK = "/content/income-tax-qa" if os.path.isdir("/content") else "/kaggle/working/income-tax-qa"
os.makedirs(WORK, exist_ok=True)

bundle = next(iter(glob.glob("/kaggle/input/**/colab_bundle.zip", recursive=True)), None)
if bundle is None:
    from google.colab import files          # Colab: upload from your laptop
    uploaded = files.upload()
    bundle = os.path.abspath(next(n for n in uploaded if n.endswith(".zip")))

with zipfile.ZipFile(bundle) as z:
    assert not any(n.endswith("/.env") for n in z.namelist()), "bundle must not contain .env"
    z.extractall(WORK)
os.chdir(os.path.join(WORK, "backend"))
print("Working in", os.getcwd()); print("PDF:", glob.glob("data/raw/*.pdf"))

## 3. Install dependencies (GPU build of onnxruntime)

In [ ]:
!pip -q uninstall -y onnxruntime
!grep -v -i "^onnxruntime" requirements.txt > /tmp/req.txt && pip -q install -r /tmp/req.txt onnxruntime-gpu
!pip -q install --no-deps -e .
import onnxruntime as ort
print("onnxruntime", ort.__version__, "providers:", ort.get_available_providers())

## 4. Build the index
Decodes the PDF, reconstructs chapters/sections, chunks, downloads the model (~436 MB) and embeds on the GPU.

In [ ]:
!python scripts/ingest.py

## 5. Sanity check: retrieval only (no LLM, no API key)

In [ ]:
from taxqa.core.config import settings
from taxqa.services.qa_pipeline import QAPipeline
p = QAPipeline(settings)
for q in ["Can I deduct interest on a housing loan for a let-out house?",
          "What deduction is available for life insurance premium?",
          "Is agricultural income taxable?"]:
    r = p.retrieve(q, use_llm=False)
    print(q); [print(f"   {h.score:.2f}  {h.chunk.label:<22} PDF {h.chunk.pages}") for h in r.context[:3]]

## 6. Package and download
Then, on your laptop from `backend/`: `python scripts/import_index.py <path>/index_bundle.zip`

In [ ]:
import zipfile, glob, os
out = os.path.abspath("index_bundle.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("data/processed/*") + glob.glob("data/index/*"):
        if not f.endswith(".gitkeep"):
            z.write(f)
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")
try:
    from google.colab import files; files.download(out)
except ImportError:
    print("Kaggle: download index_bundle.zip from the Output panel.")